# ProkBERT-mini T4 benchmark

This notebook fine-tunes the generic `neuralbioinfo/prokbert-mini` masked-language-model backbone on the canonical SeqTrainer GSE144621 shared splits. It never uses `neuralbioinfo/prokbert-mini-promoter`, never tunes on the test split, and only a completed full-data run may be entered in `Results_Final.md`.

Pinned dependencies: Hugging Face revision `feb2520a43cd9cdb5b3d8477e47209dbcb55d1dc`; official ProkBERT repository commit `8670ae92b816cff158a0b85647a8dea122e251eb`. The model license is CC-BY-NC-4.0.

In [ ]:
# 0. User-editable controls
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/SeqTrainer"
SEQTRAINER_BRANCH = "issue-3-all-model-baselines"
RUN_MODE = "full"  # "smoke" or "full"
RESUME_FROM_CHECKPOINT = True
DISCONNECT_WHEN_DONE = False

HF_MODEL_REVISION = "feb2520a43cd9cdb5b3d8477e47209dbcb55d1dc"
PROKBERT_REPOSITORY = "https://github.com/nbrg-ppcu/prokbert.git"
PROKBERT_COMMIT = "8670ae92b816cff158a0b85647a8dea122e251eb"
SEQTRAINER_COMMIT = None  # resolved from the selected branch after verifying the ProkBERT files
assert RUN_MODE in {"smoke", "full"}
if RUN_MODE == "smoke":
    print("SMOKE TEST: outputs are diagnostic only and must not be reported as benchmark results.")

## 1. Select T4 GPU

In [ ]:
import os, subprocess, sys, json, hashlib, shutil, time
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU before continuing.")
print(torch.cuda.get_device_name(0))
assert "T4" in torch.cuda.get_device_name(0), "This profile is calibrated for a 16 GB NVIDIA T4."

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_PROJECT_ROOT, exist_ok=True)

## 3. Configure project and dataset paths

In [ ]:
from pathlib import Path
LOCAL_RUN_ROOT = Path('/content/seqtrainer_prokbert_run')
LOCAL_RUN_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_PROJECT_ROOT = Path(DRIVE_PROJECT_ROOT)
DRIVE_DATA_ROOT = DRIVE_PROJECT_ROOT / 'data'
print('Drive data root:', DRIVE_DATA_ROOT)
print('Local run root:', LOCAL_RUN_ROOT)
print('Expected CSVs:', list((DRIVE_DATA_ROOT / 'promoter_classification').glob('*.csv')))

## 4. Clone the correct SeqTrainer branch

In [ ]:
SEQTRAINER_ROOT = Path('/content/SeqTrainer')
if not (SEQTRAINER_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', SEQTRAINER_BRANCH, 'https://github.com/simplyshree/SeqTrainer.git', str(SEQTRAINER_ROOT)], check=True)
print(subprocess.check_output(['git', '-C', str(SEQTRAINER_ROOT), 'branch', '--show-current'], text=True).strip())

## 5. Pin the SeqTrainer commit

In [ ]:
# Do not rewind to the pre-ProkBERT baseline; pin the selected branch commit that contains this implementation.
subprocess.run(['git', '-C', str(SEQTRAINER_ROOT), 'fetch', '--all', '--quiet'], check=True)
subprocess.run(['git', '-C', str(SEQTRAINER_ROOT), 'checkout', '--quiet', '-B', SEQTRAINER_BRANCH, f'origin/{SEQTRAINER_BRANCH}'], check=True)
resolved_seqtrainer_commit = subprocess.check_output(['git', '-C', str(SEQTRAINER_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
SEQTRAINER_COMMIT = resolved_seqtrainer_commit
required_seqtrainer_files = [SEQTRAINER_ROOT / 'src/seqtrainer/torch/prokbert_benchmark.py', SEQTRAINER_ROOT / 'notebooks/colab_benchmarks/config/prokbert_mini_t4.toml']
missing_seqtrainer_files = [str(path) for path in required_seqtrainer_files if not path.exists()]
if missing_seqtrainer_files:
    raise FileNotFoundError(f'The selected SeqTrainer branch/commit does not contain the ProkBERT implementation: {missing_seqtrainer_files}. Push or select the implementation branch.')
print('SeqTrainer commit:', resolved_seqtrainer_commit)

## 6. Install dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{SEQTRAINER_ROOT}[prokbert]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'git+{PROKBERT_REPOSITORY}@{PROKBERT_COMMIT}'], check=True)
print('Pinned ProkBERT repository:', PROKBERT_COMMIT)
import importlib.util, importlib.metadata
import prokbert
print('ProkBERT module:', prokbert.__file__)
print('ProkBERT distribution:', importlib.metadata.version('prokbert'))
print('torch.__version__:', torch.__version__)
import transformers, datasets
print('transformers.__version__:', transformers.__version__)
print('datasets.__version__:', datasets.__version__)

## 7. Print environment and GPU details

In [ ]:
print('Python:', sys.version)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('Run mode:', RUN_MODE)

## 8. Validate dataset files and hashes

In [ ]:
import pandas as pd
canonical_files = {
    'train': 'train_EP_DNA_BERT2_genomic_order.csv',
    'validation': 'eval_EP_DNA_BERT2_genomic_order.csv',
    'test': 'test_EP_DNA_BERT2_genomic_order.csv',
}
# Accept the documented SeqTrainer folder or another Drive subfolder, then report the exact paths used.
search_roots = [DRIVE_DATA_ROOT, DRIVE_PROJECT_ROOT, Path('/content/drive/MyDrive/SeqTrainer'), Path('/content/drive/MyDrive')]
split_files = {}
for split, filename in canonical_files.items():
    matches = []
    for root in search_roots:
        if root.exists():
            matches.extend(root.rglob(filename))
    matches = sorted({path.resolve() for path in matches})
    if not matches:
        searched = ', '.join(str(root) for root in search_roots)
        raise FileNotFoundError(f'Missing {split} CSV {filename}. Upload the canonical file to MyDrive/SeqTrainer/data/promoter_classification/. Searched: {searched}')
    if len(matches) > 1:
        raise RuntimeError(f'Found multiple {split} CSV files; keep one canonical copy or set DRIVE_DATA_ROOT explicitly: {matches}')
    split_files[split] = matches[0]
print('Using dataset files:')
for split, path in split_files.items():
    print(f'  {split}: {path}')
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
split_hashes = {}
split_frames = {}
for split, path in split_files.items():
    frame = pd.read_csv(path)
    if not {'sequence', 'label'}.issubset(frame.columns):
        raise ValueError(f'{split} must contain sequence and label columns: {path}')
    split_frames[split] = frame
    split_hashes[split] = sha256(path)
    print(split, 'rows=', len(frame), 'sha256=', split_hashes[split], 'lengths=', frame.sequence.astype(str).str.len().describe()[['min','50%','max']].to_dict())

## 9. Load the TOML configuration

In [ ]:
import tomllib
import sys
SRC_DIR = SEQTRAINER_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
CONFIG_PATH = SEQTRAINER_ROOT / 'notebooks/colab_benchmarks/config/prokbert_mini_t4.toml'
if not CONFIG_PATH.exists() and (SEQTRAINER_ROOT / '.git').exists():
    print('Refreshing existing SeqTrainer checkout before loading configuration...')
    subprocess.run(['git', '-C', str(SEQTRAINER_ROOT), 'fetch', '--all', '--quiet'], check=True)
    subprocess.run(['git', '-C', str(SEQTRAINER_ROOT), 'checkout', '--quiet', '-B', SEQTRAINER_BRANCH, f'origin/{SEQTRAINER_BRANCH}'], check=True)
    SRC_DIR = SEQTRAINER_ROOT / 'src'
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))
    CONFIG_PATH = SEQTRAINER_ROOT / 'notebooks/colab_benchmarks/config/prokbert_mini_t4.toml'
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f'Missing SeqTrainer configuration: {CONFIG_PATH}. Check the pinned branch/commit.')
from seqtrainer.benchmarks import load_benchmark_config
config = load_benchmark_config(CONFIG_PATH)
assert config.model.name == 'neuralbioinfo/prokbert-mini'
assert config.model.version == HF_MODEL_REVISION
assert config.model.params['tokenizer_revision'] == HF_MODEL_REVISION
assert config.evaluation.threshold_strategy == 'validation_mcc'
print('SeqTrainer source:', SRC_DIR)
print(config)

## 10. Load and audit the ProkBERT tokenizer

In [ ]:
from seqtrainer.torch.prokbert_benchmark import load_prokbert_backbone, audit_prokbert_tokenizer
tokenizer, encoder = load_prokbert_backbone(
    config.model.name, revision=HF_MODEL_REVISION, tokenizer_revision=HF_MODEL_REVISION,
    trust_remote_code=True, local_files_only=False,
)
audit = audit_prokbert_tokenizer(tokenizer, encoder, split_frames['train'].sequence.head(5).tolist(), model_max_length=512)
assert audit['attention_mask_present']
assert audit['rows'] >= 5
print(audit)

## 11. Load the ProkBERT encoder

In [ ]:
hidden_size = getattr(encoder.config, 'hidden_size', getattr(encoder.config, 'd_model', None))
if hidden_size is None:
    raise ValueError('Could not infer encoder hidden size')
print('Encoder:', type(encoder).__name__, 'hidden_size=', hidden_size)
print('Promoter-finetuned checkpoint used: False')

## 12. Run a forward-pass and memory preflight

In [ ]:
sample = tokenizer(split_frames['train'].sequence.head(16).tolist(), padding='longest', truncation=True, max_length=512, return_attention_mask=True, return_tensors='pt')
encoder = encoder.cuda().eval()
with torch.no_grad():
    sample_out = encoder(input_ids=sample['input_ids'].cuda(), attention_mask=sample['attention_mask'].cuda())
hidden = sample_out.last_hidden_state if hasattr(sample_out, 'last_hidden_state') else sample_out[0]
print('input_ids:', tuple(sample['input_ids'].shape), 'encoder output:', tuple(hidden.shape))
assert hidden.ndim == 3 and hidden.shape[0] == sample['input_ids'].shape[0]
print('Peak memory MiB:', round(torch.cuda.max_memory_allocated() / 2**20, 1))
torch.cuda.reset_peak_memory_stats()

## 13. Train using validation-MCC checkpointing

In [ ]:
from dataclasses import replace
from seqtrainer.torch.prokbert_benchmark import run_prokbert_csv_splits
run_config = config
run_base = SEQTRAINER_ROOT
if RUN_MODE == 'smoke':
    smoke_dir = LOCAL_RUN_ROOT / 'smoke_data'
    smoke_dir.mkdir(parents=True, exist_ok=True)
    smoke_paths = {}
    for split, frame in split_frames.items():
        sampled = frame.groupby('label', group_keys=False).apply(lambda group: group.sample(n=min(len(group), 16), random_state=42)).reset_index(drop=True)
        path = smoke_dir / f'{split}.csv'
        sampled.to_csv(path, index=False)
        smoke_paths[split] = str(path)
    run_config = replace(run_config, dataset=replace(run_config.dataset, split_files=smoke_paths), training=replace(run_config.training, max_epochs=1), outputs=replace(run_config.outputs, output_dir=str(LOCAL_RUN_ROOT)))
    run_base = None
else:
    run_config = replace(run_config, dataset=replace(run_config.dataset, split_files={split: str(path) for split, path in split_files.items()}), outputs=replace(run_config.outputs, output_dir=str(LOCAL_RUN_ROOT)))
resume_path = LOCAL_RUN_ROOT / 'best_checkpoint.pt'
if not RESUME_FROM_CHECKPOINT or not resume_path.exists():
    resume_path = None
if resume_path is not None:
    run_config = replace(run_config, training=replace(run_config.training, params={**run_config.training.params, 'resume_from_checkpoint': str(resume_path)}))
result = run_prokbert_csv_splits(run_config, base_dir=run_base, output_dir=LOCAL_RUN_ROOT, tokenizer=tokenizer, encoder=encoder)
print('status:', result.status, 'output:', result.output_dir)

## 14. Reload the best checkpoint

In [ ]:
best_checkpoint = LOCAL_RUN_ROOT / 'best_checkpoint.pt'
if not best_checkpoint.exists():
    raise FileNotFoundError(best_checkpoint)
checkpoint = torch.load(best_checkpoint, map_location='cpu', weights_only=False)
assert checkpoint['model_revision'] == HF_MODEL_REVISION
assert checkpoint['split_hashes'] == (split_hashes if RUN_MODE == 'full' else checkpoint['split_hashes'])
print('best epoch:', checkpoint['epoch'], 'best validation MCC:', checkpoint['best_validation_mcc'])

## 15. Select the final threshold from validation

In [ ]:
final_threshold = float(checkpoint['selected_validation_threshold'])
print('Validation-selected threshold:', final_threshold)
print('The threshold is stored in the best checkpoint and was selected from validation probabilities only.')

## 16. Evaluate the held-out test split

In [ ]:
metrics = json.loads((LOCAL_RUN_ROOT / 'metrics.json').read_text())
if RUN_MODE == 'smoke':
    print('SMOKE TEST ONLY — these metrics are not scientific benchmark results.')
else:
    print('Final held-out test metrics:', json.dumps(metrics['test'], indent=2, sort_keys=True))

## 17. Save metrics and reproducibility artifacts

In [ ]:
required = ['metrics.csv','metrics.json','predictions.csv','manifest.json','history.csv','best_checkpoint.pt','tokenizer','run_summary.txt']
for name in required:
    path = LOCAL_RUN_ROOT / name
    if not path.exists():
        raise FileNotFoundError(f'Missing required artifact: {path}')
print('All local artifacts are present.')
manifest = json.loads((LOCAL_RUN_ROOT / 'manifest.json').read_text())
print('Manifest revision:', manifest['model']['metadata']['model_revision'])

## 18. Display learning curves and confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
history = pd.read_csv(LOCAL_RUN_ROOT / 'history.csv')
predictions = pd.read_csv(LOCAL_RUN_ROOT / 'predictions.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.epoch, history.train_loss, label='train loss')
axes[0].plot(history.epoch, history.validation_loss, label='validation loss')
axes[0].set_title('Loss'); axes[0].legend()
test = predictions[predictions.split == 'test']
ConfusionMatrixDisplay.from_predictions(test.label, test.prediction, ax=axes[1], colorbar=False)
axes[1].set_title('Held-out test confusion matrix')
plt.show()

## 19. Copy all outputs to Google Drive

In [ ]:
drive_output = DRIVE_PROJECT_ROOT / 'outputs' / 'benchmarks' / f'prokbert_mini_t4_{RUN_MODE}'
if RUN_MODE == 'smoke':
    drive_output = DRIVE_PROJECT_ROOT / 'outputs' / 'smoke_tests' / 'prokbert_mini_t4_smoke_test'
drive_output.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_RUN_ROOT, drive_output, dirs_exist_ok=True)
print('Drive output:', drive_output)

## 20. Final verification

In [ ]:
for name in required:
    assert (drive_output / name).exists(), f'Missing Drive artifact: {drive_output / name}'
drive_manifest = json.loads((drive_output / 'manifest.json').read_text())
assert drive_manifest['model']['metadata']['base_model'] == 'neuralbioinfo/prokbert-mini'
assert drive_manifest['model']['metadata']['promoter_finetuned_checkpoint_used'] is False
assert drive_manifest['model']['metadata']['model_revision'] == HF_MODEL_REVISION
print('Final Drive paths:')
for name in required:
    print(drive_output / name)
if RUN_MODE == 'smoke':
    print('SMOKE TEST COMPLETE — do not copy metrics into Results_Final.md.')
else:
    print('FULL RUN COMPLETE — only now may the pending Results_Final.md row be updated with measured metrics.')

## 21. Optional runtime disconnect

Smoke outputs are diagnostic only. Set `RUN_MODE = "smoke"` for that path. Only a completed full-data run on the canonical shared splits may be entered in `Results_Final.md`. A full run must finish copying all artifacts before disconnection.

In [ ]:
# Final cell: verify, copy, print Drive paths, then optionally disconnect.
for name in required:
    assert (LOCAL_RUN_ROOT / name).exists(), f'Missing required local artifact: {LOCAL_RUN_ROOT / name}'
drive_output.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_RUN_ROOT, drive_output, dirs_exist_ok=True)
for name in required:
    assert (drive_output / name).exists(), f'Missing required Drive artifact: {drive_output / name}'
print('Final Drive paths:')
for name in required:
    print(drive_output / name)
if RUN_MODE == 'smoke':
    print('SMOKE TEST COMPLETE — do not copy metrics into Results_Final.md.')
from google.colab import runtime
if DISCONNECT_WHEN_DONE:
    runtime.unassign()